# 09 — Feature engineering: all-vehicle baseline (builds on notebook 08)

**This is now the active line of development**, per direct instruction — notebook 08 (all 4
vehicles, raw features, stratified split, logistic regression) is the baseline; this notebook adds
rolling/lag and calendar features on top of it, the same way notebook 02 did for the single-vehicle
rebuild, but now for all 4 vehicles merged.

**One detail carried forward correctly despite the different split strategy:** rolling/lag features
still have to be computed *within each vehicle's own timeline* (`groupby("user_profile")`), even
though the final train/test split is stratified/random rather than chronological. Feature
engineering and the train/test split are separate decisions — computing a rolling mean across two
different vehicles' rows would be wrong regardless of how the split itself is done later.


In [1]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RAW_SENSOR_COLS = ["SOC", "SOH", "Charging_Cycles", "Battery_Temp", "Motor_RPM", "Motor_Torque",
                    "Motor_Temp", "Brake_Pad_Wear", "Charging_Voltage", "Tire_Pressure"]
SEQ_SENSORS = ["Motor_RPM", "Motor_Torque", "Motor_Temp", "Battery_Temp"]
ROLL_WINDOWS = [6, 12, 24]
LAGS = [1, 3, 6]


## 1. Load the all-vehicle cleaned dataset (from notebook 08)

In [2]:
df = pd.read_csv("../data/processed/all_vehicles_cleaned.csv", parse_dates=["timestamp"])
df = df.sort_values(["user_profile", "timestamp"]).reset_index(drop=True)
print(f"{len(df):,} rows across {df['user_profile'].nunique()} vehicles")


175,176 rows across 4 vehicles


## 2. Rolling / lag sensor features — computed per vehicle

In [3]:
seq_feature_cols = []

for col in SEQ_SENSORS:
    grouped = df.groupby("user_profile")[col]
    for w in ROLL_WINDOWS:
        mean_col, std_col = f"{col}_roll_mean_{w}h", f"{col}_roll_std_{w}h"
        df[mean_col] = grouped.transform(lambda s, w=w: s.rolling(window=w, min_periods=w).mean())
        df[std_col] = grouped.transform(lambda s, w=w: s.rolling(window=w, min_periods=w).std())
        seq_feature_cols += [mean_col, std_col]
    for lag in LAGS:
        lag_col = f"{col}_lag_{lag}h"
        df[lag_col] = grouped.shift(lag)
        seq_feature_cols.append(lag_col)
    delta_col = f"{col}_delta_6h"
    df[delta_col] = df[col] - grouped.shift(6)
    seq_feature_cols.append(delta_col)

print(f"Added {len(seq_feature_cols)} rolling/lag features across {len(SEQ_SENSORS)} sensors.")


Added 40 rolling/lag features across 4 sensors.


In [4]:
# Sanity check: no cross-vehicle bleed — each vehicle's first rows should be NaN until enough
# history exists within that vehicle specifically.
check = df.groupby("user_profile").head(2)[["user_profile", "timestamp", "Motor_RPM_roll_mean_24h", "Motor_RPM_lag_6h"]]
check


,user_profile,timestamp,Motor_RPM_roll_mean_24h,Motor_RPM_lag_6h
0,daily_user,2020-01-01 00:00:00,NaN,NaN
1,daily_user,2020-01-01 01:00:00,NaN,NaN
43794,heavy_user,2020-01-01 00:00:00,NaN,NaN
43795,heavy_user,2020-01-01 01:00:00,NaN,NaN
87588,moderate_user,2020-01-01 00:00:00,NaN,NaN
87589,moderate_user,2020-01-01 01:00:00,NaN,NaN
131382,rare_user,2020-01-01 00:00:00,NaN,NaN
131383,rare_user,2020-01-01 01:00:00,NaN,NaN


## 3. Calendar features

In [5]:
df["hour_of_day"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month_of_year"] = df["timestamp"].dt.month
df["hours_since_start"] = (
    df.groupby("user_profile")["timestamp"]
      .transform(lambda t: (t - t.min()).dt.total_seconds() / 3600)
)

seasonality_check = pd.DataFrame({
    "by_hour_range_pp": [df.groupby("hour_of_day")["is_fault"].mean().mul(100).agg(lambda s: s.max() - s.min())],
    "by_dow_range_pp": [df.groupby("day_of_week")["is_fault"].mean().mul(100).agg(lambda s: s.max() - s.min())],
    "by_month_range_pp": [df.groupby("month_of_year")["is_fault"].mean().mul(100).agg(lambda s: s.max() - s.min())],
}, index=["issue-rate spread (percentage points)"])
seasonality_check


,by_hour_range_pp,by_dow_range_pp,by_month_range_pp
issue-rate spread (percentage points),3.712276,0.973235,0.413754


In [6]:
CALENDAR_FEATURE_COLS = ["hour_of_day", "day_of_week", "month_of_year", "hours_since_start"]
print("Calendar features carried forward (final selection happens in notebook 10):")
print(CALENDAR_FEATURE_COLS)


Calendar features carried forward (final selection happens in notebook 10):
['hour_of_day', 'day_of_week', 'month_of_year', 'hours_since_start']


## 4. Drop cold-start rows (incomplete rolling/lag window), keep target NaNs documented

In [7]:
before = len(df)
feature_na_mask = df[seq_feature_cols].isna().any(axis=1)
print(f"Dropping {feature_na_mask.sum()} cold-start rows ({feature_na_mask.sum() / df['user_profile'].nunique():.0f} per vehicle).")
df = df.loc[~feature_na_mask].reset_index(drop=True)
print(f"{before:,} rows -> {len(df):,} rows after dropping cold-start rows.")


Dropping 92 cold-start rows (23 per vehicle).


175,176 rows -> 175,084 rows after dropping cold-start rows.


## 5. Save the engineered all-vehicle table + candidate feature list

In [8]:
RAW_AND_SEQ_COLS = RAW_SENSOR_COLS + seq_feature_cols
CANDIDATE_FEATURE_COLS = RAW_AND_SEQ_COLS + CALENDAR_FEATURE_COLS

with open("../data/processed/candidate_feature_cols_all_vehicles.json", "w") as f:
    json.dump(CANDIDATE_FEATURE_COLS, f, indent=2)

df.to_csv("../data/processed/all_vehicles_features.csv", index=False)

print(f"Saved {len(df):,} rows x {len(df.columns)} columns to ../data/processed/all_vehicles_features.csv")
print(f"Saved {len(CANDIDATE_FEATURE_COLS)} candidate feature names to "
      f"../data/processed/candidate_feature_cols_all_vehicles.json")
print("(Candidate list, not final — feature selection happens in notebook 10.)")


Saved 175,084 rows x 60 columns to ../data/processed/all_vehicles_features.csv
Saved 54 candidate feature names to ../data/processed/candidate_feature_cols_all_vehicles.json
(Candidate list, not final — feature selection happens in notebook 10.)


## 6. What this notebook establishes, going into notebook 10

1. Rolling/lag features built correctly per-vehicle, same discipline as the single-vehicle rebuild,
   despite the different final split strategy.
2. Calendar features added, with the same seasonality evidence check as before, now computed across
   all 4 vehicles.
3. A candidate feature list is saved, explicitly not final — notebook 10 does real feature selection
   (MI + permutation importance, checked against a random-noise benchmark) before anything gets
   called a model input.
